In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e8/sample_submission.csv
/kaggle/input/playground-series-s5e8/train.csv
/kaggle/input/playground-series-s5e8/test.csv


In [2]:
train_path = '/kaggle/input/playground-series-s5e8/train.csv'
test_path = '/kaggle/input/playground-series-s5e8/test.csv'
sample_path = pd.read_csv('/kaggle/input/playground-series-s5e8/sample_submission.csv')
# read the data and store data in DataFrame 
train_data = pd.read_csv(train_path) 
test_data = pd.read_csv(test_path)
# print a summary of the data 
train_data.describe()
print("Full train dataset shape is {}".format(train_data.shape))
print("Test shape:", test_data.shape)

Full train dataset shape is (750000, 18)
Test shape: (250000, 17)


In [3]:
print(f'Missing values for training data: {train_data.isnull().sum()}')
print(f'Missing values for test data: {test_data.isnull().sum()}')

Missing values for training data: id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64
Missing values for test data: id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
dtype: int64


In [4]:
train_data.drop(['id', 'job', 'day', 'month'], axis=1, inplace = True)
test_data.drop(['job', 'day', 'month'], axis=1, inplace = True)

In [5]:
train_data.head()

,age,marital,education,default,balance,housing,loan,contact,duration,campaign,pdays,previous,poutcome,y
0,42,married,secondary,no,7,no,no,cellular,117,3,-1,0,unknown,0
1,38,married,secondary,no,514,no,no,unknown,185,1,-1,0,unknown,0
2,36,married,secondary,no,602,yes,no,unknown,111,2,-1,0,unknown,0
3,27,single,secondary,no,34,yes,no,unknown,10,2,-1,0,unknown,0
4,26,married,secondary,no,889,yes,no,cellular,902,1,-1,0,unknown,1


In [6]:
test_data.head()

,id,age,marital,education,default,balance,housing,loan,contact,duration,campaign,pdays,previous,poutcome
0,750000,32,married,secondary,no,1397,yes,no,unknown,224,1,-1,0,unknown
1,750001,44,married,tertiary,no,23,yes,no,cellular,586,2,-1,0,unknown
2,750002,36,married,primary,no,46,yes,yes,cellular,111,2,-1,0,unknown
3,750003,58,married,secondary,no,-1380,yes,yes,unknown,125,1,-1,0,unknown
4,750004,28,single,secondary,no,1950,yes,no,cellular,181,1,-1,0,unknown


In [7]:
#check duplicates

print(train_data.duplicated().sum())
print(test_data.duplicated().sum())

14480
0


In [8]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 14 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   age        750000 non-null  int64 
 1   marital    750000 non-null  object
 2   education  750000 non-null  object
 3   default    750000 non-null  object
 4   balance    750000 non-null  int64 
 5   housing    750000 non-null  object
 6   loan       750000 non-null  object
 7   contact    750000 non-null  object
 8   duration   750000 non-null  int64 
 9   campaign   750000 non-null  int64 
 10  pdays      750000 non-null  int64 
 11  previous   750000 non-null  int64 
 12  poutcome   750000 non-null  object
 13  y          750000 non-null  int64 
dtypes: int64(7), object(7)
memory usage: 80.1+ MB


In [9]:
train_data.describe()

,age,balance,duration,campaign,pdays,previous,y
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,40.926395,1204.067397,256.229144,2.577008,22.412733,0.298545,0.120651
std,10.098829,2836.096759,272.555662,2.718514,77.319998,1.335926,0.325721
min,18.000000,-8019.000000,1.000000,1.000000,-1.000000,0.000000,0.000000
25%,33.000000,0.000000,91.000000,1.000000,-1.000000,0.000000,0.000000
50%,39.000000,634.000000,133.000000,2.000000,-1.000000,0.000000,0.000000
75%,48.000000,1390.000000,361.000000,3.000000,-1.000000,0.000000,0.000000
max,95.000000,99717.000000,4918.000000,63.000000,871.000000,200.000000,1.000000


In [10]:
print(train_data.columns)

Index(['age', 'marital', 'education', 'default', 'balance', 'housing', 'loan',
       'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'y'],
      dtype='object')


In [11]:
Y = train_data.y

In [12]:
featr_to_encode = ['marital', 'marital', 'education', 'default', 'housing', 
                  'loan', 'contact', 'poutcome']
le = LabelEncoder()
for i in featr_to_encode:
    train_data[i] = le.fit_transform(train_data[i])
    test_data[i] = le.fit_transform(test_data[i])

train_data.head()
X = train_data[featr_to_encode]
test_X = test_data[featr_to_encode]

In [13]:
print(X.describe())

             marital        marital      education        default  \
count  750000.000000  750000.000000  750000.000000  750000.000000   
mean        1.160569       1.160569       1.227461       0.017132   
std         0.577240       0.577240       0.705607       0.129763   
min         0.000000       0.000000       0.000000       0.000000   
25%         1.000000       1.000000       1.000000       0.000000   
50%         1.000000       1.000000       1.000000       0.000000   
75%         2.000000       2.000000       2.000000       0.000000   
max         2.000000       2.000000       3.000000       1.000000   

             housing           loan        contact       poutcome  
count  750000.000000  750000.000000  750000.000000  750000.000000  
mean        0.548384       0.139969       0.659963       2.756635  
std         0.497654       0.346955       0.917652       0.764445  
min         0.000000       0.000000       0.000000       0.000000  
25%         0.000000       0.000000   

In [14]:
print(X.head())

   marital  marital  education  default  housing  loan  contact  poutcome
0        1        1          1        0        0     0        0         3
1        1        1          1        0        0     0        2         3
2        1        1          1        0        1     0        2         3
3        2        2          1        0        1     0        2         3
4        1        1          1        0        1     0        0         3


In [15]:
from sklearn.tree import DecisionTreeRegressor

# Define model. Specify a number for random_state to ensure same results each run
loan_model = DecisionTreeRegressor(random_state=1)

# Fit model
loan_model.fit(X, Y)

DecisionTreeRegressor(random_state=1)

In [16]:
print("Making predictions for the following 5 reason:")
print(X.head())
print("The predictions are")
print(loan_model.predict(X.head()))

Making predictions for the following 5 reason:
   marital  marital  education  default  housing  loan  contact  poutcome
0        1        1          1        0        0     0        0         3
1        1        1          1        0        0     0        2         3
2        1        1          1        0        1     0        2         3
3        2        2          1        0        1     0        2         3
4        1        1          1        0        1     0        0         3
The predictions are
[0.14609295 0.03968479 0.03554566 0.05973112 0.0807584 ]


In [17]:
from sklearn.metrics import mean_absolute_error

predicted_loan_prices = loan_model.predict(X)
mean_absolute_error(Y, predicted_loan_prices)

0.18182223594118205

In [18]:
from sklearn.model_selection import train_test_split

# split data into training and validation data, for both features and target
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time we
# run this script.
train_X, val_X, train_Y, val_Y = train_test_split(X, Y, random_state = 0)
# Define model
loan_model = DecisionTreeRegressor()
# Fit model
loan_model.fit(train_X, train_Y)

# get predicted prices on validation data
val_predictions = loan_model.predict(val_X)
val_mae=mean_absolute_error(val_Y, val_predictions)
print(mean_absolute_error(val_Y, val_predictions))
print("Validation MAE: {:,.0f}".format(val_mae))
test_preds = loan_model.predict(test_X)

0.18188524728952232
Validation MAE: 0


In [19]:
output = pd.DataFrame({'id': test_data.id,
                       'y': test_preds})
output.to_csv('submission.csv', index=False)
print(output)

            id         y
0       750000  0.034791
1       750001  0.091809
2       750002  0.038873
3       750003  0.023946
4       750004  0.126070
...        ...       ...
249995  999995  0.091809
249996  999996  0.095941
249997  999997  0.784585
249998  999998  0.032067
249999  999999  0.262149

[250000 rows x 2 columns]
